# T5/T8 DGX batch

The queue contains two matched pairs at seed 17. Run CPU preparation first. Each cell starts a fresh process and uses durable receipts. Keep the companion files beside this notebook and use a fresh session label for an allocation. Both job cells, smoke, restoration and backup are initially disabled. Read the batch guide for exact private inputs and approvals.


## 1. Verify the exact operational checkout


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")

import fcntl
import shutil
from tempfile import mkdtemp

def git(directory, *args):
    return subprocess.check_output(["git", "-C", str(directory), *args], env=env, text=True).strip()

def verify_checkout(directory):
    if git(directory, "rev-parse", "HEAD") != pin or git(directory, "status", "--porcelain", "--ignored", "--untracked-files=all"):
        raise RuntimeError("Expected clean exact checkout; inspect existing source")

with (root / ".t5-t8-source.lock").open("a+b") as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    if not source.exists():
        if shutil.disk_usage(root).free < 512 * 2**20:
            raise RuntimeError("Insufficient storage for source checkout")
        temporary = Path(mkdtemp(prefix="t5-t8-source-", dir=root))
        git(temporary, "init", "--quiet")
        git(temporary, "fetch", "--no-tags", "--depth=1", "https://github.com/Akshai0007/Thesi_Tokenizer_Pipeline.git", pin)
        git(temporary, "checkout", "--detach", "--quiet", "FETCH_HEAD")
        verify_checkout(temporary)
        if source.exists():
            raise RuntimeError("Source path appeared during checkout")
        temporary.rename(source)
    verify_checkout(source)
print("Verified operational source", pin, source)

import sys
from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)


run("source", interpreter=sys.executable)


## 2. Verify the pinned Python environment


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)


run("environment")


## 3. Restore the frozen inputs from Package B


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



restore_approved = False
archive = root / "MOM-BPE-DGX-B-four-arm-a874e17.zip"
if restore_approved:
    run("restore", "--enable", "--archive", str(archive))
else:
    print("Archive restoration disabled. Skip this cell when the exact data already exist.")


## 4. Verify input SHA-256 hashes and historical references


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("verify")


## 5. Verify all four exposure plans


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("plan")


## 6. Check CPU readiness and private scratch


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)


run("ready", "--scope", "all")


## 7. Inspect physical GPUs and scratch


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("inspect")


## 8. Run an approved A100 and telemetry diagnostic


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



smoke_approved = False
allocated = False
shared_approved = False
uuid = ""
scope = "first"  # Explicitly choose "all" to approve both pairs for this session.
if smoke_approved and allocated:
    sharing = ["--shared"] if shared_approved else []
    run("smoke", "--enable", "--allocated", "--uuid", uuid, "--scope", scope, *sharing)
else:
    print("GPU diagnostic disabled. Enter the approved physical A100 allocation first.")


## 9. Run T5 then T8 with fixed source bytes


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



enable_training = False
if enable_training:
    run("first", "--enable")
else:
    print("Training disabled. A successful authorized smoke must cover this pair.")


## 10. Verify and compare the first pair


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)


run("results", "--scope", "first")
run("analyze")


## 11. Run T5 then T8 with fixed training tokens


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



enable_training = False
if enable_training:
    run("second", "--enable")
else:
    print("Training disabled. A successful authorized smoke must cover this pair.")


## 12. Verify all four result and checkpoint identities


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("results", "--scope", "all")


## 13. Generate protocol-specific scientific comparisons


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("analyze")


## 14. Back up verified artifacts to an approved private mount


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



backup_approved = False
destination = ""
scope = "all"
kind = "all"  # Or choose an individual artifact category from the guide.
if backup_approved:
    if not destination:
        raise ValueError("Supply the approved backup mount")
    run("backup", "--enable", "--destination", destination, "--scope", scope, "--backup-kind", kind)
else:
    print("Backup disabled. Supply a separately approved private destination.")


## 15. Write the final session report


In [ ]:
from pathlib import Path
import os
import subprocess

python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1",
           GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("MLTOK_PLUGINS", None)
root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
pin = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + pin)
if source.resolve() != source:
    raise RuntimeError("Unexpected source symlink")


from hashlib import sha256

companions = {'dgx_batch.py': '7446f65dae7def3ed08cd8cf1fde42544188abb4add22cd0422eaf24c045e16e', 't5_t8_batch_v1.json': 'd0bfb6b3994cff4ca2c0159718e0a29da91e1f8aa029f5fa3d7169541905a5b7', 't5_t8_dgx.py': '2141360763bc86c3aafaa7c4a2e66b5ee301a3b1a1d7564a9f10a3b67f4f2a0a', 't5_t8_inputs_v1.json': '4266825c1132cb44555313b0b68cd489bbfe55566a508be2a8ed606a0bad7750'}
folder = Path.cwd()

def run(action, *options, interpreter=python):
    for name, digest in companions.items():
        path = folder / name
        if path.is_symlink() or sha256(path.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Missing or inconsistent companion {name}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(interpreter), "-I", "-B", str(folder / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--session", session, *options],
                   cwd=source, env=env, check=True)



run("report")
